In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc PyGithub networkx qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Iskay Quantum Optimizer – Eine Qiskit Function von Kipu Quantum

> **Note:** * Qiskit Functions sind ein experimentelles Feature, das ausschließlich Nutzern des IBM Quantum&reg; Premium Plan, Flex Plan und On-Prem (über die IBM Quantum Platform API) zur Verfügung steht. Sie befinden sich im Preview-Status und können sich noch ändern.

## Übersicht
Mit dem Iskay Quantum Optimizer von Kipu Quantum kannst du komplexe Optimierungsprobleme auf IBM&reg; Quantencomputern lösen. Dieser Solver nutzt Kipus hochmodernen [bf-DCQO-Algorithmus](https://doi.org/10.48550/arXiv.2409.04477), der lediglich die Zielfunktion als Eingabe benötigt und Problemlösungen automatisch liefert. Er kann Optimierungsprobleme mit bis zu 156 Qubits verarbeiten und ermöglicht damit die Nutzung aller Qubits der IBM-Quantengeräte. Der Optimizer verwendet ein 1-zu-1-Mapping zwischen klassischen Variablen und Qubits, sodass du Optimierungsprobleme mit bis zu 156 binären Variablen lösen kannst.

Der Optimizer ermöglicht die Lösung unbeschränkter binärer Optimierungsprobleme. Neben der gängigen QUBO-Formulierung (Quadratic Unconstrained Binary Optimization) unterstützt er auch Optimierungsprobleme höherer Ordnung (HUBO). Der Solver verwendet einen nicht-variationellen Quantenalgorithmus, der den Großteil der Berechnungen auf Quantengeräten durchführt.

Im Folgenden findest du weitere Details zum verwendeten Algorithmus sowie eine kurze Anleitung zur Nutzung der Funktion und Benchmarking-Ergebnisse für verschiedene Probleminstanzen unterschiedlicher Größe und Komplexität.
## Beschreibung
Der Optimizer ist eine einsatzbereite Implementierung modernster Quantenoptimierungsalgorithmen. Er löst Optimierungsprobleme, indem er hochkomprimierte Quantenschaltkreise auf Quantenhardware ausführt. Diese Kompression wird durch das Einführen kontradiabatischer Terme in die zugrundeliegende Zeitentwicklung des Quantensystems erreicht. Der Algorithmus führt mehrere Iterationen von Hardware-Läufen durch, um die endgültigen Lösungen zu erhalten, und kombiniert diese mit Nachbearbeitung. Diese Schritte sind nahtlos in den Workflow des Optimizers integriert und werden automatisch ausgeführt.

### Wie funktioniert der Quantum Optimizer?
Dieser Abschnitt erläutert die Grundlagen des implementierten bf-DCQO-Algorithmus. Eine Einführung in den Algorithmus findest du auch auf dem [Qiskit-YouTube-Kanal.](https://www.youtube.com/watch?v=33QmsXhIlpU&t=1223s)

Der Algorithmus basiert auf der Zeitentwicklung eines Quantensystems, das sich im Laufe der Zeit transformiert, wobei die Problemlösung im Grundzustand des Quantensystems am Ende der Entwicklung kodiert ist. Gemäß dem [adiabatischen Theorem](https://en.wikipedia.org/wiki/Adiabatic_theorem) muss diese Entwicklung langsam verlaufen, damit das System in seinem Grundzustand bleibt. Die Digitalisierung dieser Entwicklung ist die Grundlage der digitisierten Quanten-Adiabatik-Berechnung (DQA) und des bekannten QAOA-Algorithmus. Die erforderliche langsame Entwicklung ist jedoch für wachsende Problemgrößen nicht praktikabel, da sie zu einer zunehmenden Schaltkreistiefe führt. Durch den Einsatz kontradiabatischer Protokolle kannst du unerwünschte Anregungen während kurzer Entwicklungszeiten unterdrücken und gleichzeitig im Grundzustand bleiben. Die Digitalisierung dieser kürzeren Entwicklungszeit führt zu Quantenschaltkreisen mit geringerer Tiefe und weniger verschränkenden Gates.

Die Schaltkreise der bf-DCQO-Algorithmen verwenden typischerweise bis zu zehnmal weniger verschränkende Gates als DQA und drei- bis viermal weniger verschränkende Gates als Standard-QAOA-Implementierungen. Aufgrund der geringeren Anzahl von Gates treten bei der Schaltkreisausführung auf Hardware weniger Fehler auf. Daher benötigt der Optimizer keine Techniken wie Fehlerunterdrückung oder Fehlerminderung. Die Implementierung dieser Techniken in zukünftigen Versionen kann die Lösungsqualität noch weiter verbessern.

Obwohl der bf-DCQO-Algorithmus Iterationen verwendet, ist er nicht-variationell. Nach jeder Iteration des Algorithmus wird die Zustandsverteilung gemessen. Die erhaltene Verteilung wird verwendet, um ein sogenanntes Bias-Feld zu berechnen. Das Bias-Feld ermöglicht es, die nächste Iteration von einem Energiezustand nahe der zuvor gefundenen Lösung zu starten. Auf diese Weise bewegt sich der Algorithmus mit jeder Iteration zu Lösungen niedrigerer Energie. Typischerweise sind etwa zehn Iterationen ausreichend, um zu einer Lösung zu konvergieren, was insgesamt deutlich weniger Iterationen erfordert als variationelle Algorithmen, die in der Größenordnung von etwa 100 Iterationen liegen.

Der Optimizer kombiniert den bf-DCQO-Algorithmus mit klassischer Nachbearbeitung. Nach dem Messen der Zustandsverteilung wird eine lokale Suche durchgeführt. Bei der lokalen Suche werden die Bits der gemessenen Lösung zufällig umgekippt. Nach dem Umkippen wird die Energie des neuen Bitstrings bewertet. Wenn die Energie niedriger ist, wird der Bitstring als neue Lösung beibehalten. Die lokale Suche skaliert nur linear mit der Anzahl der Qubits und ist daher rechnerisch günstig. Da die Nachbearbeitung lokale Bitflips korrigiert, kompensiert sie Bit-Flip-Fehler, die häufig aus Hardware-Unvollkommenheiten und Auslesefehlern resultieren.

### Workflow
Es folgt ein Schema des Workflows des Quantum Optimizers.

![Workflow](../docs/images/guides/kipu-optimization/workflow.svg "Workflow des Quantum Optimizers")

Durch die Verwendung des Quantum Optimizers lässt sich die Lösung eines Optimierungsproblems auf Quantenhardware auf Folgendes reduzieren:
* Formulierung der Zielfunktion des Problems
* Zugriff auf den Optimizer über Qiskit Functions
* Ausführen des Optimizers und Einsammeln des Ergebnisses
## Benchmarks
Die nachstehenden Benchmark-Kennzahlen zeigen, dass der Optimizer Probleme mit bis zu 156 Qubits effektiv bewältigt, und bieten einen allgemeinen Überblick über die Genauigkeit und Skalierbarkeit des Optimizers für verschiedene Problemtypen. Beachte, dass die tatsächlichen Leistungskennzahlen je nach den spezifischen Problemmerkmalen variieren können, wie z. B. die Anzahl der Variablen, die Dichte und Lokalität der Terme in der Zielfunktion sowie die Polynomordnung.

Die folgende Tabelle enthält das Approximationsverhältnis (AR), eine Metrik, die wie folgt definiert ist:
$$
AR = \frac{C^{*} - C_\textrm{max}}{C_{\textrm{min}} - C_{\textrm{max}}},
$$
wobei $C$ die Zielfunktion ist, $C_{\textrm{min}}$, $C_{\textrm{max}}$ ihre minimalen und maximalen Werte sind und $C^{*}$ die Kosten der besten gefundenen Lösung sind. Daher bedeutet AR=100\%, dass der Grundzustand des Problems gefunden wurde.

| Beispiel          | Anzahl der Qubits | Approximationsverhältnis | Gesamtzeit (s) | Laufzeitnutzung (s) | Gesamtanzahl Shots | Anzahl Iterationen |
| ----------------- | :---------------: | :----------------------: | :------------: | :-----------------: | :-----------------: | :----------------: |
| Ungewichtetes MaxCut |        28        |   100%   |      180       |        30         |          30k          |          5           |
| Ungewichtetes MaxCut |        30        |   100%   |      180       |        30         |          30k          |          5           |
| Ungewichtetes MaxCut |        32        |   100%   |      180       |        30         |          30k          |          5           |
| Ungewichtetes MaxCut |        80        |   100%   |      480       |        60         |          90k          |          9           |
| Ungewichtetes MaxCut |       100        |   100%   |      330       |        60         |          60k          |          6           |
| Ungewichtetes MaxCut |       120        |   100%   |      370       |        60         |          60k          |          6           |
| HUBO 1            |       156        |   100%   |      600       |        70         |         100k          |          10          |
| HUBO 2            |       156        |   100%   |      600       |        70         |         100k          |          10          |

- Die MaxCut-Instanzen mit 28, 30 und 32 Qubits wurden auf ibm_sherbrooke ausgeführt. Instanzen mit 80, 100 und 120 wurden auf einem Heron-r2-Prozessor ausgeführt.
- Die HUBO-Instanzen wurden ebenfalls auf einem Heron-r2-Prozessor ausgeführt.

Alle Benchmark-Instanzen sind auf GitHub zugänglich (siehe [Kipu-Benchmark-Instanzen](https://github.com/Kipu-Quantum-GmbH/benchmark-instances)). Ein Beispiel zur Ausführung dieser Instanzen findest du in [Beispiel 3: Benchmark-Instanzen](#example-3-benchmark-instances).
## Eingaben und Ausgaben
### Eingabe
In der folgenden Tabelle findest du alle Eingabeparameter, die der Quantum Optimizer akzeptiert. Der nachfolgende Abschnitt **Optionen** geht näher auf die verfügbaren `options` ein.
| Name         | Typ                | Beschreibung                                                                                                                                                                              | Erforderlich | Standard                                                                                      | Beispiel                                                                                                             |
| ------------ | ------------------ | :---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------ | --------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------------- |
| problem      | `Dict[str, float]` | Die Koeffizienten des Optimierungsproblems in QUBO/HUBO- oder Spin-Format. Weitere Informationen zur Problemspezifikation findest du unter [Akzeptierte Problemformate](#accepted-problem-formats) | Ja           | N/A                                                                                           | `{"()": -21.0, "(0, 4)": 0.5,"(0, 2)": 0.5,"(0, 1)": 0.5,"(1, 3)": 0.5}`                                             |
| problem_type | `str`              | Gibt an, ob die Problemkoeffizienten im binären (QUBO/HUBO) oder Spin-Format vorliegen. Die zwei Möglichkeiten sind `"spin"` oder `"binary"`                                              | Ja           | N/A                                                                                           | `"spin"`                                                                                                             |
| backend_name | `str`              | Name des Backends, an das die Anfrage gestellt werden soll                                                                                                                                | Ja           | N/A                                                      | `"ibm_fez"`                                                                                                          |                                                                                         |
| options      | `Dict[str, Any]`   | Optionen zur Steuerung der Hardware-Übermittlung, z. B. Anzahl der Shots. Weitere Details zur Optionskonfiguration findest du im Abschnitt [Optionen](#options) | Nein         | Standardwerte der Optionskonfiguration sind im Abschnitt [Optionen](#options) aufgeführt       | `{"shots": 5000, "num_iterations": 3, "use_session": True, "seed_transpiler": 42}` |
#### Akzeptierte Problemformate
Die Argumente `problem` und `problem_type` kodieren ein Optimierungsproblem der Form

$$
\begin{align}
\min_{(x_1, x_2, \ldots, x_n) \in D} C(x_1, x_2, \ldots, x_n) \nonumber
\end{align}
$$
wobei

$$
C(x_1, ... , x_n) = a + \sum_{i} b_i x_i + \sum_{i, j} c_{i, j} x_i x_j + ... + \sum_{k_1, ..., k_m} g_{k_1, ..., k_m} x_{k_1} ... x_{k_m}
$$

- Durch die Wahl von `problem_type = "binary"` gibst du an, dass die Kostenfunktion im `binary`-Format vorliegt, d. h. $D = {0,  1}^{n}$, die Kostenfunktion also in QUBO/HUBO-Formulierung geschrieben ist.
- Durch die Wahl von `problem_type = "spin"` hingegen ist die Kostenfunktion in Ising-Formulierung geschrieben, wobei $D = {-1, 1}^{n}$.

Die Koeffizienten des Problems sollen in einem Dictionary wie folgt kodiert werden:
$$
\begin{align} \nonumber
&\texttt{{} \\ \nonumber
&\texttt{"()"}&: \quad &a, \\ \nonumber
&\texttt{"(i,)"}&: \quad &b_i, \\ \nonumber
&\texttt{"(i, j)"}&: \quad &c_{i, j}, \\ \nonumber
&\quad  \vdots \\ \nonumber
&\texttt{"(} k_1, ..., k_m  \texttt{)"} &: \quad &g_{k_1, ..., k_m}, \\ \nonumber
&\texttt{}}
\end{align}
$$

- Bitte beachte, dass die Schlüssel des Dictionarys Zeichenketten sein müssen, die ein gültiges Tupel nicht-wiederholender ganzer Zahlen enthalten.
#### Optionen
Iskay bietet Feinabstimmungsmöglichkeiten durch optionale Parameter. Während die Standardwerte für die meisten Probleme gut funktionieren, kannst du das Verhalten für spezifische Anforderungen anpassen:
| Parameter | Typ | Standard | Beschreibung |
|-----------|-----|----------|--------------|
| **shots** | `int` | 10000 | Quantenmessungen pro Iteration (höher = genauer) |
| **num_iterations** | `int` | 10 | Algorithmus-Iterationen (mehr Iterationen können die Lösungsqualität verbessern) |
| **use_session** | `bool` | True | IBM-Sessions für kürzere Wartezeiten verwenden |
| **seed_transpiler** | `int` | None | Für reproduzierbare Quantenschaltkreis-Kompilierung setzen |
| **direct_qubit_mapping** | `bool` | False | Virtuelle Qubits direkt auf physische Qubits abbilden |
| **job_tags** | `List[str]` | None | Benutzerdefinierte Tags für die Job-Verfolgung |
| **preprocessing_level** | `int` | 0 | Intensität der Problemvorverarbeitung (0–3) – siehe Details unten |
| **postprocessing_level** | `int` | 2 | Lösungsverfeinerungsstufe (0–2) – siehe Details unten |
| **transpilation_level** | `int` | 0 | Transpiler-Optimierungsversuche (0–5) – siehe Details unten |
| **transpile_only** | `bool` | False | Schaltkreisoptimierung analysieren, ohne die vollständige Ausführung zu starten |

**Vorverarbeitungsstufen (0–3)**: Besonders wichtig für größere Probleme, die derzeit nicht in die Kohärenzzeiten der Hardware passen. Höhere Vorverarbeitungsstufen erzielen geringere Schaltkreistiefen durch Näherungen bei der Problem-Transpilation:
- **Stufe 0**: Exakt, längere Schaltkreise
- **Stufe 1**: Gute Balance zwischen Genauigkeit und Näherung; nur Gates mit Winkeln im untersten 10. Perzentil werden entfernt
- **Stufe 2**: Etwas höhere Näherung; Gates mit Winkeln im untersten 20. Perzentil werden entfernt und `approximation_degree=0.95` bei der Transpilation verwendet
- **Stufe 3**: Maximale Näherungsstufe; Gates im untersten 30. Perzentil werden entfernt und `approximation_degree=0.90` bei der Transpilation verwendet

**Transpilationsstufen (0–5)**: Steuern die erweiterten Transpiler-Optimierungsversuche für die Quantenschaltkreis-Kompilierung. Dies kann zu einem erhöhten klassischen Overhead führen, und in manchen Fällen ändert sich die Schaltkreistiefe möglicherweise nicht. Der Standardwert `2` führt in der Regel zum kleinsten Schaltkreis und ist relativ schnell.
- **Stufe 0**: Optimierung des zerlegten DCQO-Schaltkreises (Layout, Routing, Scheduling)
- **Stufe 1**: Optimierung von `PauliEvolutionGate` und dann des zerlegten DCQO-Schaltkreises (max_trials=10)
- **Stufe 2**: Optimierung von `PauliEvolutionGate` und dann des zerlegten DCQO-Schaltkreises (max_trials=15)
- **Stufe 3**: Optimierung von `PauliEvolutionGate` und dann des zerlegten DCQO-Schaltkreises (max_trials=20)
- **Stufe 4**: Optimierung von `PauliEvolutionGate` und dann des zerlegten DCQO-Schaltkreises (max_trials=25)
- **Stufe 5**: Optimierung von `PauliEvolutionGate` und dann des zerlegten DCQO-Schaltkreises (max_trials=50)

**Nachverarbeitungsstufen (0–2)**: Steuern den Umfang der klassischen Optimierung, die Bit-Flip-Fehler durch unterschiedliche Anzahl gieriger Durchläufe einer lokalen Suche kompensiert:
- **Stufe 0**: 1 Durchlauf
- **Stufe 1**: 2 Durchläufe
- **Stufe 2**: 3 Durchläufe

**Transpile-only-Modus**: Jetzt für Benutzer verfügbar, die die Schaltkreisoptimierung analysieren möchten, ohne die vollständige Quantenalgorithmus-Ausführung zu starten.

**Beispiel einer benutzerdefinierten Konfiguration**: So könntest du Iskay mit verschiedenen Einstellungen konfigurieren:

In [1]:
custom_options = {
    "shots": 15_000,  # Higher shot count for better statistics
    "num_iterations": 12,  # More iterations for solution refinement
    "preprocessing_level": 1,  # Light preprocessing for problem simplification
    "postprocessing_level": 2,  # Maximum postprocessing for solution quality
    "transpilation_level": 3,  # Using higher transpilation level for circuit optimization
    "seed_transpiler": 42,  # Fixed seed for reproducible results
    "job_tags": ["custom_config"],  # Custom tracking tags
}

**Seed-Optimierung**: Beachte, dass `seed_transpiler` standardmäßig auf `None` gesetzt ist. Dies aktiviert den automatischen Optimierungsprozess des Transpilers. Bei `None` startet das System einen Versuch mit mehreren Seeds und wählt denjenigen aus, der die beste Schaltkreistiefe erzeugt, wobei die volle Leistung des Parameters `max_trials` für jede Transpilationsstufe genutzt wird.

**Transpilationsstufen-Performance**: Eine Erhöhung der Anzahl von `max_trials` mit höheren Werten für `transpilation_level` erhöht unweigerlich die Transpilationszeit, ändert aber möglicherweise nicht immer den endgültigen Schaltkreis – dies hängt stark von der spezifischen Schaltkreisstruktur und -komplexität ab. Bei manchen Schaltkreisen/Problemen kann der Unterschied zwischen 10 Versuchen (Stufe 1) und 50 Versuchen (Stufe 5) jedoch erheblich sein, sodass das Erkunden dieser Parameter der Schlüssel zum erfolgreichen Finden einer Lösung sein kann.
### Ausgabe
| Name   | Typ                | Beschreibung                                                           | Beispiel                                                               |
| ------ | ------------------ | ---------------------------------------------------------------------- | ---------------------------------------------------------------------- |
| result | `Dict[str, Any]` | Lösung und Metadaten. Die Struktur variiert je nach Option `transpile_only`. | Siehe „Inhalt des Ergebnis-Dictionarys" unten      |
### Inhalt des Ergebnis-Dictionarys
Die Struktur des Ergebnis-Dictionarys hängt vom Ausführungsmodus ab:

| Feld          | Typ                | Modus | Beschreibung                                                                                                             | Beispiel |
| ------------ | :------------------: | :--: | ------------------------------------------------------------------------------------------------------------------------ | ------- |
| **solution**     | `Dict[str, int]` | **Standard** | Die sortierte gemappte Lösung, bei der die Schlüssel Variablenindizes (als Zeichenketten) numerisch sortiert sind und die Werte die entsprechenden Variablenwerte sind (1/-1 für Spin-Probleme, 1/0 für binäre Probleme).| `{'0': -1, '1': -1, '2': -1, '3': 1, '4': 1}` |
| **solution_info**      | `Dict[str, Any]`   | **Standard** | Detaillierte Informationen zur Lösung (siehe Details unten)                                           | `{'bitstring': '11100', 'cost': -13.8, 'seed_transpiler': 42, 'mapping': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}}` |
| **prob_type** | `str`             | **Standard** | Der Typ des Optimierungsproblems ('spin' oder 'binary')                                                           | `'spin'` |
| **transpilation_info** | `Dict[str, Any]` | **Nur transpilieren** | Schaltkreisanalyse und Transpilationsdetails (siehe Details unten) | `{'best_seed': 42, 'transpilation_time_seconds': 50.06, 'transpiled_circuit': {'depth': 576, 'gate_count': 4177, 'num_qubits': 156, 'width': 176, 'operations': {'sx': 1325, 'rx': 891, 'cz': 783, 'rz': 650, 'rzz': 466, 'x': 42, 'measure': 20}}}` |

#### Standardausführung
Wenn der optionale Parameter `transpile_only=False`:

**solution_info-Dictionary:**
- **"bitstring"** (`str`): Die rohe Bitstring-Darstellung der Lösung.
- **"cost"** (`float`): Der Kosten-/Energiewert der Lösung.
- **"seed_transpiler"** (`int`): Der zufällige Seed, der für den Transpiler verwendet wurde und dieses Ergebnis erzeugt hat.
- **"mapping"** (`Dict[int, int]`): Das ursprüngliche Qubit-zu-Variable-Mapping, das in der Berechnung verwendet wurde.
- **"qpu_time"** (`float`, optional): Die QPU-Ausführungszeit in Sekunden.

**Hinweise zum Variablen-Mapping:**
- Das `solution`-Dictionary wird aus dem Lösungs-Bitstring unter Verwendung des `mapping`-Objekts für die Indizierung der Variablen gewonnen.
- Bei `problem_type=spin` wird die Zuweisung $1 \rightarrow -1, \quad 0 \rightarrow 1$ verwendet.
- Schlüssel im Lösungs-Dictionary sind Variablenindizes, numerisch als Zeichenketten sortiert.

#### Transpilationsanalyse
Wenn der optionale Parameter `transpile_only=True`:

**transpilation_info-Dictionary:**
- **"best_seed"** (`int`): Der optimale für die Transpilation gefundene Seed
- **"transpilation_time_seconds"** (`float`): Dauer des Transpilationsprozesses
- **"transpiled_circuit"** (`Dict`): Schaltkreisanalyse mit:
  - **"depth"** (`int`): Schaltkreistiefe (Anzahl der Schichten)
  - **"gate_count"** (`int`): Gesamtanzahl der Gates im Schaltkreis
  - **"num_qubits"** (`int`): Anzahl der verwendeten Qubits
  - **"width"** (`int`): Schaltkreisbreite
  - **"operations"** (`Dict[str, int]`): Anzahl jedes verwendeten Gate-Typs

**Verwendung des Transpile-only-Modus:**
- Verfügbar für Benutzer, die die Schaltkreisoptimierung analysieren möchten, ohne die vollständige Quantenalgorithmus-Ausführung zu starten.
- Nützlich für Schaltkreisanalysen, Tiefenoptimierungsstudien und das Verstehen von Transpilationseffekten vor dem Starten der vollständigen Ausführung.
## Erste Schritte
In dieser Dokumentation gehen wir die Schritte zur Verwendung des Iskay Quantum Optimizers durch. Dabei zeigen wir kurz, wie du die Funktion aus dem Catalog lädst und dein Problem in eine gültige Eingabe umwandelst, und zeigen gleichzeitig, wie du mit verschiedenen optionalen Parametern experimentieren kannst.

Ein detaillierteres Beispiel findest du im Tutorial [Das Market-Split-Problem mit dem Iskay Quantum Optimizer von Kipu Quantum lösen](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer), in dem wir den gesamten Prozess der Verwendung des Iskay Solvers zur Lösung des Market-Split-Problems durcharbeiten, das eine reale Ressourcenzuteilungsaufgabe darstellt, bei der Märkte in ausgewogene Verkaufsregionen aufgeteilt werden müssen, um genaue Nachfrageziele zu erfüllen.

Authentifiziere dich mit deinem API-Schlüssel, den du im [IBM Quantum Platform-Dashboard](http://quantum.cloud.ibm.com/) findest, und wähle die Qiskit Function wie folgt aus:

In [ ]:
# ruff: noqa: F821

> **Note:** Der folgende Code setzt voraus, dass du deine Zugangsdaten gespeichert hast. Falls nicht, folge den Anweisungen unter [IBM Cloud-Konto speichern](/guides/functions#install-qiskit-functions-catalog-client), um dich mit deinem API-Schlüssel zu authentifizieren.

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(
    channel="ibm_quantum_platform",
    instance="INSTANCE_CRN",
    token="YOUR_API_KEY",  # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
)

# Access Function
optimizer = catalog.load("kipu-quantum/iskay-quantum-optimizer")

## Beispiel 1: Einfache Kostenfunktion

Betrachte die Kostenfunktion in Spin-Formulierung:
$$
C(x_0, x_1, x_2, x_3, x_4) = 1 + 1.5x_0 + 2x_1 + 1.3x_2 + 2.5x_0x_3 + 3.5x_1x_4 + 4x_0x_1x_2
$$

wobei $(x_0, ..., x_4) \in {-1, 1}^5$.

Die Lösung dieser einfachen Kostenfunktion ist
$$
(x_0, x_1, x_2, x_3, x_4) = (-1, -1, -1, 1, 1)
$$
mit dem Minimalwert $C^{*} = -6$

### 1. Zielfunktion erstellen

Wir beginnen mit der Erstellung eines Dictionarys mit den Koeffizienten der Zielfunktion wie folgt:

In [ ]:
objective_func = {
    "()": 1,
    "(0,)": 1.5,
    "(1,)": 2,
    "(2,)": 1.3,
    "(0, 3)": 2.5,
    "(1, 4)": 3.5,
    "(0, 1, 2)": 4,
}

### 2. Optimizer ausführen
Wir lösen das Problem, indem wir den Optimizer ausführen. Da $(x_0, ..., x_4) \in {-1, 1}^5$ müssen wir `problem_type=spin` setzen.

In [ ]:
# Setup options to run the optimizer
options = {"shots": 5000, "num_iterations": 5, "use_session": True}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": backend_name,  # such as "ibm_fez"
    "options": options,
}

job = optimizer.run(**arguments)

### 3. Ergebnis abrufen
Die Lösung des Optimierungsproblems wird direkt vom Optimizer geliefert.

In [ ]:
print(job.result())

Dies zeigt ein Dictionary der folgenden Form:

```
{'solution': {'0': -1, '1': -1, '2': -1, '3': 1, '4': 1},
 'solution_info': {'bitstring': '11100',
  'cost': -13.8,
  'seed_transpiler': 42,
  'mapping': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}},
 'prob_type': 'spin'}
```

Beachte, dass das Dictionary `solution` den Ergebnisvektor $(x_0, x_1, x_2, x_3, x_4) = (-1, -1, -1, 1, 1)$ anzeigt.

## Beispiel 2: MaxCut

Viele Graphprobleme wie MaxCut oder Maximum Independent Set sind NP-schwere Probleme und ideale Kandidaten zum Testen von Quantenalgorithmen und Hardware. Dieses Beispiel demonstriert die Lösung des MaxCut-Problems eines 3-regulären Graphen mit dem Quantum Optimizer.

Um dieses Beispiel auszuführen, musst du zusätzlich zur `qiskit-ibm-catalog` das Paket `networkx` installieren. Führe dazu folgenden Befehl aus:

In [ ]:
# %pip install networkx numpy

### 1. Zielfunktion erstellen
Beginne mit der Generierung eines zufälligen 3-regulären Graphen. Für diesen Graphen definieren wir die Zielfunktion des MaxCut-Problems.

In [ ]:
import networkx as nx

# Create a random 3-regular graph
G = nx.random_regular_graph(3, 10, seed=42)


# Create the objective function for MaxCut in Ising formulation
def graph_to_ising_maxcut(G):
    """
    Convert a NetworkX graph to an Ising Hamiltonian for the Max-Cut problem.
    Args:
        G (networkx.Graph): The input graph.
    Returns:
        dict: The objective function of the Ising model
    """
    # Initialize the linear and quadratic coefficients
    objective_func = {}
    # Populate the coefficients
    for i, j in G.edges:
        objective_func[f"({i}, {j})"] = 0.5
    return objective_func


objective_func = graph_to_ising_maxcut(G)

### 2. Optimizer ausführen
Löse das Problem, indem du den Optimizer ausführst.

In [ ]:
options = {"shots": 5000, "num_iterations": 5, "use_session": True}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": backend_name,  # such as "ibm_fez"
    "options": options,
}

job = optimizer.run(**arguments)

### 3. Ergebnis abrufen
Rufe das Ergebnis ab und bilde den Lösungs-Bitstring auf die ursprünglichen Graphknoten zurück.

In [ ]:
print(job.result())

Die Lösung des MaxCut-Problems ist direkt im Unter-Dictionary `solution` des Ergebnisobjekts enthalten

In [ ]:
maxcut_solution = job.result()["solution"]

## Beispiel 3: Benchmark-Instanzen
Die Benchmark-Instanzen sind auf GitHub verfügbar: [Kipu-Benchmark-Instanzen.](https://github.com/Kipu-Quantum-GmbH/benchmark-instances)

Die Instanzen können mit der Bibliothek `pygithub` geladen werden. Führe dazu folgenden Befehl aus:

In [ ]:
# %pip install pygithub

Die Pfade für die Benchmark-Instanzen sind:

**Maxcut:**
- `'maxcut/maxcut_28_nodes.json'`
- `'maxcut/maxcut_30_nodes.json'`
- `'maxcut/maxcut_32_nodes.json'`
- `'maxcut/maxcut_80_nodes.json'`
- `'maxcut/maxcut_100_nodes.json'`
- `'maxcut/maxcut_120_nodes.json'`

**HUBO:**
- `'HUBO/hubo1_marrakesh.json'`
- `'HUBO/hubo2_marrakesh.json'`

Um die Performance des Benchmarks für die HUBO-Instanzen zu reproduzieren, wähle das Backend `ibm_marrakesh` und setze `direct_qubit_mapping` in dem Unter-Dictionary `options` auf `True`.
Das folgende Beispiel führt die MaxCut-Instanz mit 32 Knoten aus.

In [ ]:
from github import Github
import urllib
import json
import ast

repo = "Kipu-Quantum-GmbH/benchmark-instances"
path = "maxcut/maxcut_32_nodes.json"
gh = Github()
repo = gh.get_repo(repo)
branch = "main"
file = repo.get_contents(urllib.parse.quote(path), ref=branch)

# load json file with benchmark problem
problem_json = json.loads(file.decoded_content)

# convert objective function to compatible format
objective_func = {
    key: ast.literal_eval(value) for key, value in problem_json.items()
}


# Setup configuration to run the optimizer
options = {
    "shots": 5_000,
    "num_iterations": 5,
    "use_session": True,
    "direct_qubit_mapping": False,
}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": "ibm_brisbane",
    "options": options,
}

job = optimizer.run(**arguments)

result = job.result()

## Anwendungsfälle
Typische Anwendungsfälle für den Optimierungs-Solver sind kombinatorische Optimierungsprobleme. Du kannst Probleme aus vielen Branchen wie Finanzen, Pharmazeutik oder Logistik lösen. Einige Beispiele folgen.
* Portfolio-Optimierung (QUBO): [wissenschaftliche Publikation](https://doi.org/10.1103/PhysRevApplied.22.054037) und [Whitepaper](https://kipu-quantum.com/zope64/kipu_2024/content/e3915/e3916/e4187/White-Paper-2-Financial-modeling-on-quantum-computers-using-digitally-compressed-algorithms-1.pdf)
* Proteinfaltung (HUBO): [wissenschaftliche Publikation](https://doi.org/10.1103/PhysRevApplied.20.014024)
* Logistikplanung (QUBO): [wissenschaftliche Publikation](https://doi.org/10.1103/PhysRevApplied.22.064068)
* Netzwerkoptimierung: [Webinar](https://www.youtube.com/watch?v=w5SrCIK88No)
* Market Split (QUBO): [Tutorial](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer)

Falls du an einem konkreten Anwendungsfall interessiert bist und ein dediziertes Mapping entwickeln möchtest, können wir dich dabei unterstützen. [Kontaktiere uns.](https://share-eu1.hsforms.com/2Ff8cgWvTR9ukT_fPoaNhDw2dqpz5)
## Support erhalten
Für Support wende dich an support@kipu-quantum.com.
## Nächste Schritte
- [Zugang zum Quantum Optimizer von Kipu Quantum anfragen](https://share-eu1.hsforms.com/2Ff8cgWvTR9ukT_fPoaNhDw2dqpz5)
- Probiere das Tutorial [Das Market-Split-Problem mit dem Iskay Quantum Optimizer von Kipu Quantum lösen](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer) aus.
- Lies [Romero, S. V., et al. (2025).  Bias-Field Digitized Counterdiabatic Quantum Algorithm for Higher-Order Binary Optimization. arXiv preprint arXiv:2409.04477.](https://arxiv.org/abs/2409.04477)
- Lies [Cadavid, A. G., et al. (2024).  Bias-field digitized counterdiabatic quantum optimization. arXiv preprint arXiv:2405.13898.](https://arxiv.org/abs/2405.13898)
- Lies [Chandarana, P., et al. (2025).  Runtime Quantum Advantage with Digital Quantum Optimization. arXiv preprint arXiv:2505.08663.](https://arxiv.org/abs/2505.08663)
## Zusätzliche Informationen
Iskay ist, wie unser Firmenname Kipu Quantum, ein peruanisches Wort. Obwohl wir ein Startup aus Deutschland sind, stammen diese Wörter aus dem Heimatland eines unserer Mitgründer, wo der Quipu vor etwa 2000 Jahren v. Chr. als eine der allerersten von der Menschheit entwickelten Rechenmaschinen galt.